# 15. Sorting, Searching & Set Operations (5+ Years Interview Guide)
Exhaustive revision guide to sorting, index sorting (argsort), binary search (searchsorted), where, extract, and set algebra on raw_transactions.csv.

### Key 5-Year Interview Concepts Covered:
- **Sorting & Finding**: Dedicated cell for `np.sort()`, `np.argsort()`, and `np.searchsorted()`.
- **Conditional Extraction**: Dedicated cell for `np.where()` and `np.extract()`.
- **Mathematical Set Operations**: Dedicated cell for `np.unique()`, `np.intersect1d()`, `np.union1d()`, and `np.setdiff1d()`.

This interactive revision guide loads and operates directly on `data/raw_transactions.csv` using dedicated cells per method.

In [1]:
# Setup imports & dataset loading from raw_transactions.csv
import numpy as np
import pandas as pd
import sys
import time
import os

# Load raw transactions and extract aligned NumPy numeric arrays
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
raw_df = pd.read_csv(csv_path)
clean_raw = raw_df.dropna(subset=['transaction_amount', 'is_fraud', 'account_age_months']).reset_index(drop=True)
amounts = clean_raw['transaction_amount'].to_numpy(dtype=np.float64)
fraud_flags = clean_raw['is_fraud'].to_numpy(dtype=np.int8)
account_ages = clean_raw['account_age_months'].to_numpy(dtype=np.float32)

print(f"NumPy Version: {np.__version__}")
print(f"Loaded from {csv_path} ({len(amounts)} clean aligned rows):")
print(f"- amounts array: shape {amounts.shape}, dtype {amounts.dtype}")
print(f"- fraud_flags array: shape {fraud_flags.shape}, dtype {fraud_flags.dtype}")
print(f"- account_ages array: shape {account_ages.shape}, dtype {account_ages.dtype}")

NumPy Version: 1.26.4
Loaded from data/raw_transactions.csv (14262 clean aligned rows):
- amounts array: shape (14262,), dtype float64
- fraud_flags array: shape (14262,), dtype int8
- account_ages array: shape (14262,), dtype float32


### Sorting: `np.sort()` vs In-Place `.sort()`
**Explanation**: Sorts transaction amounts ascending.

**Syntax**: `np.sort(amounts)`

In [2]:
sorted_amounts = np.sort(amounts)
print('Smallest 3 Transactions:', sorted_amounts[:3])
print('Largest 3 Transactions:', sorted_amounts[-3:])

Smallest 3 Transactions: [5.08 5.39 5.6 ]
Largest 3 Transactions: [1999.74 1999.85 1999.98]


### Indirect Index Sorting with `np.argsort()`
**Explanation**: Returns indices that sort transaction amounts, allowing synchronized ordering of customer IDs.

**Syntax**: `sort_idx = np.argsort(amounts)`

In [3]:
sort_idx = np.argsort(amounts)
print('Top 3 Highest Amount Indices:', sort_idx[-3:])
print('Corresponding Customer IDs:', clean_raw['customer_id'].to_numpy()[sort_idx[-3:]])

Top 3 Highest Amount Indices: [ 6986  4819 11445]
Corresponding Customer IDs: ['C22224' 'C30635' 'C22899']


### Binary Search on Sorted Arrays: `np.searchsorted()`
**Explanation**: Uses binary search in $O(\log N)$ time to find spending bracket bucket positions.

**Syntax**: `np.searchsorted(brackets, amounts[:5])`

In [4]:
brackets = np.array([50.0, 200.0, 500.0, 1000.0])
bucket_positions = np.searchsorted(brackets, amounts[:5])
print('Assigned Bracket Buckets for First 5 Transactions:', bucket_positions)

Assigned Bracket Buckets for First 5 Transactions: [4 2 1 1 4]


### Conditional Selection with `np.where()`
**Explanation**: Flags transactions > $500 as 'HIGH' or 'NORMAL'.

**Syntax**: `np.where(amounts > 500, 'HIGH', 'NORMAL')`

In [5]:
risk_labels = np.where(amounts[:5] > 500.0, 'HIGH', 'NORMAL')
print('Categorized Risk Labels (first 5):', risk_labels)

Categorized Risk Labels (first 5): ['HIGH' 'NORMAL' 'NORMAL' 'NORMAL' 'HIGH']


### Conditional Extraction with `np.extract()`
**Explanation**: Extracts all fraudulent transaction amounts directly.

**Syntax**: `np.extract(fraud_flags == 1, amounts)`

In [6]:
fraud_amounts = np.extract(fraud_flags == 1, amounts)
print(f'Extracted {len(fraud_amounts)} Fraudulent Amounts (first 5):', fraud_amounts[:5].round(2))

Extracted 1580 Fraudulent Amounts (first 5): [1805.16  927.5  1893.8  1944.85 1593.81]


### Set Uniqueness with `np.unique()`
**Explanation**: Extracts distinct card types and their frequency counts.

**Syntax**: `np.unique(raw_df['card_type'], return_counts=True)`

In [7]:
cards = clean_raw['card_type'].dropna().to_numpy()
uniques, counts = np.unique(cards, return_counts=True)
print('Unique Card Types:', uniques)
print('Occurrence Counts:', counts)

Unique Card Types: ['Amex' 'Discover' 'MasterCard' 'Visa']
Occurrence Counts: [3594 3546 3587 3535]


### Set Intersection: `np.intersect1d()`
**Explanation**: Finds customer IDs that transacted in both North America and Europe.

**Syntax**: `np.intersect1d(custs_na, custs_eu)`

In [8]:
custs_na = clean_raw[clean_raw['region'] == 'North America']['customer_id'].unique()
custs_eu = clean_raw[clean_raw['region'] == 'Europe']['customer_id'].unique()
shared_custs = np.intersect1d(custs_na, custs_eu)
print('Customers Transacting in both NA and Europe Count:', len(shared_custs))

Customers Transacting in both NA and Europe Count: 0


### Set Union: `np.union1d()`
**Explanation**: Returns all distinct customer IDs across regions.

**Syntax**: `np.union1d(custs_na, custs_eu)`

In [9]:
print('Total Unique Customers across both NA & Europe:', len(np.union1d(custs_na, custs_eu)))

Total Unique Customers across both NA & Europe: 0


### Set Difference: `np.setdiff1d()`
**Explanation**: Finds customers exclusive to North America.

**Syntax**: `np.setdiff1d(custs_na, custs_eu)`

In [10]:
exclusive_na = np.setdiff1d(custs_na, custs_eu)
print('Customers Exclusive to North America Count:', len(exclusive_na))

Customers Exclusive to North America Count: 0


## Section: Senior Fintech Interview Scenarios (5+ Years Experience)

### Q1: Top-K Highest Value Transactions in $O(N)$ Time with `np.argpartition`
**Explanation**: Find the top 5 largest transaction amounts in $O(N)$ linear time without running a full sort.

**Syntax**: `top_k = np.argpartition(amounts, -5)[-5:]`

In [11]:
top5_idx = np.argpartition(amounts, -5)[-5:]
top5_sorted_idx = top5_idx[np.argsort(amounts[top5_idx])[::-1]]
print('Top 5 Largest Transactions in O(N) Time ($):', amounts[top5_sorted_idx].round(2))

Top 5 Largest Transactions in O(N) Time ($): [1999.98 1999.85 1999.74 1999.52 1999.36]
